In [ ]:
#"""Quotation Model Training (Module 2)"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import joblib
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load data
df = pd.read_csv('../data/synthetic/quotation_data.csv')
print(f"Loaded {len(df)} records")
df.head()

In [ ]:
# ============================================================================
# 1. FEATURE ENGINEERING
# ============================================================================

def preprocess_quotation_data(df):
    """Preprocess data for quotation model"""
    
    df_processed = df.copy()
    
    # Encode categorical variables
    le_vehicle_type = LabelEncoder()
    le_fuel_type = LabelEncoder()
    le_city_tier = LabelEncoder()
    le_make = LabelEncoder()
    
    df_processed['vehicle_type_encoded'] = le_vehicle_type.fit_transform(df_processed['vehicle_type'])
    df_processed['fuel_type_encoded'] = le_fuel_type.fit_transform(df_processed['fuel_type'])
    df_processed['city_tier_encoded'] = le_city_tier.fit_transform(df_processed['city_tier'])
    df_processed['make_encoded'] = le_make.fit_transform(df_processed['vehicle_make'])
    
    # Create additional features
    # 1. Age of vehicle (already have vehicle_age)
    # 2. IDV to Age ratio
    df_processed['idv_age_ratio'] = df_processed['idv'] / (df_processed['vehicle_age'] + 1)
    
    # 3. Premium per IDV
    df_processed['premium_per_idv'] = df_processed['premium'] / df_processed['idv']
    
    # 4. NCB discount impact
    df_processed['ncb_impact'] = df_processed['ncb_percent'] / 100
    
    return df_processed, {
        'vehicle_type': le_vehicle_type,
        'fuel_type': le_fuel_type,
        'city_tier': le_city_tier,
        'make': le_make
    }

df_processed, encoders = preprocess_quotation_data(df)

# Features for model
feature_columns = [
    'vehicle_type_encoded', 'fuel_type_encoded', 'city_tier_encoded',
    'make_encoded', 'manufacturing_year', 'idv', 'ncb_percent',
    'vehicle_age', 'idv_age_ratio'
]

X = df_processed[feature_columns]
y = df_processed['premium']

print(f"Features: {X.columns.tolist()}")
print(f"Target: Premium (mean: ₹{y.mean():.2f})")

In [ ]:
# ============================================================================
# 2. TRAIN-TEST SPLIT
# ============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

In [ ]:

# ============================================================================
# 3. SCALE FEATURES
# ============================================================================

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler
joblib.dump(scaler, '../models/scaler.pkl')

In [ ]:
# ============================================================================
# 4. TRAIN MODEL
# ============================================================================

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_scaled, y_train)

In [ ]:
# ============================================================================
# 5. EVALUATE MODEL
# ============================================================================

y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

# Metrics
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

print("=" * 60)
print("MODEL PERFORMANCE METRICS")
print("=" * 60)
print(f"Train R² Score: {train_r2:.4f}")
print(f"Test R² Score: {test_r2:.4f}")
print(f"Train RMSE: ₹{train_rmse:.2f}")
print(f"Test RMSE: ₹{test_rmse:.2f}")
print(f"Train MAE: ₹{train_mae:.2f}")
print(f"Test MAE: ₹{test_mae:.2f}")

In [ ]:
# ============================================================================
# 6. SHAP EXPLANATIONS
# ============================================================================

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_scaled[:100])

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_scaled[:100], 
                  feature_names=feature_columns,
                  show=False)
plt.tight_layout()
plt.savefig('../models/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTOP FEATURES BY IMPORTANCE:")
print(feature_importance.head(10))

In [ ]:
# ============================================================================
# 7. SAVE MODEL
# ============================================================================

os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/quotation_model.pkl')

# Save encoders
joblib.dump(encoders, '../models/quotation_encoders.pkl')

print("\n✅ Model saved successfully!")
print(f"Model file: ../models/quotation_model.pkl")
print(f"Encoders file: ../models/quotation_encoders.pkl")

In [ ]:
# ============================================================================
# 8. TEST PREDICTION
# ============================================================================

def predict_premium(vehicle_data, model, encoders, scaler, feature_columns):
    """Make a premium prediction"""
    
    # Encode categorical variables
    vehicle_type_encoded = encoders['vehicle_type'].transform([vehicle_data['vehicle_type']])[0]
    fuel_type_encoded = encoders['fuel_type'].transform([vehicle_data['fuel_type']])[0]
    city_tier_encoded = encoders['city_tier'].transform([vehicle_data['city_tier']])[0]
    make_encoded = encoders['make'].transform([vehicle_data['vehicle_make']])[0]
    
    # Prepare features
    features = np.array([[
        vehicle_type_encoded,
        fuel_type_encoded,
        city_tier_encoded,
        make_encoded,
        vehicle_data['manufacturing_year'],
        vehicle_data['idv'],
        vehicle_data['ncb_percent'],
        vehicle_data['vehicle_age'],
        vehicle_data['idv'] / (vehicle_data['vehicle_age'] + 1)
    ]])
    
    # Scale
    features_scaled = scaler.transform(features)
    
    # Predict
    prediction = model.predict(features_scaled)[0]
    return prediction

# Test prediction
test_vehicle = {
    'vehicle_type': 'car',
    'vehicle_make': 'Maruti',
    'fuel_type': 'Petrol',
    'city_tier': 'tier1',
    'manufacturing_year': 2020,
    'idv': 500000,
    'ncb_percent': 20,
    'vehicle_age': 4
}

predicted = predict_premium(test_vehicle, model, encoders, scaler, feature_columns)
print(f"\nTest Prediction:")
print(f"Vehicle: {test_vehicle['vehicle_make']} {test_vehicle['vehicle_type']}")
print(f"Predicted Premium: ₹{predicted:.2f}")

print("\n✅ Model training complete!")